# Phase 2ii: Iterative TreeTime Clock Filtering & Down Sampling

Generates a temporal tree using TreeTime with **iterative clock filtering**:
TreeTime is run in a loop, removing outlier sequences each iteration until no
more outliers are detected (convergence). If `downsample_to` is provided,
downsampling is performed after convergence.

<details>
    <summary>Click To See A Description of Parameters</summary>
        <pre>
            <code>
save_dir: str
    Path to directory for saving outputs in.

fasta_path: str, optional
    Path to fasta file containing sequences.

metadata_path: str, optional
    Path to csv or tsv containing metadata.

sample_id_field: str, default 'strain'
    Name of field in metadata_path containing IDs matching fasta_path.

collection_date_field: str, default 'date'
    Name of field in metadata_path containing collection dates (YYYY-MM-DD).

root_strain_names: list of strings, optional
    IDs of sequences used to root the tree. Excluded from outlier removal
    during clock filtering. Pruned after convergence if remove_root=True.

downsample_to: int, optional
    If provided, downsample the tree after clock filter convergence.

tree_dir_name: str
    Name of directory housing IQ-TREE outputs and TreeTime outputs.

seed: int, optional
    Seed for TreeTime and downsampling.

clock_filter: float or None, default 3.0
    Threshold for clock filtering (z-score for local, n_iqd for residual).
    Set to None to disable (single run, no iteration).

clock_filter_method: str, default 'local'
    Method: 'local' (z-score) or 'residual' (IQD).

max_iterations: int, default 50
    Maximum clock filter iterations before stopping.
            </code>
        </pre>
</details>


In [ ]:
save_dir = None
fasta_path = None
metadata_path = None
sample_id_field = 'strain'
collection_date_field = 'date'
root_strain_names = None
remove_root = True
downsample_to = None
tree_dir_name = None
seed = None
clock_filter = 3.0
clock_filter_method = 'local'
max_iterations = 50
remove_future_tips = True
clock_std = None


### Import packages and get data if not in save_dir

In [ ]:
from beast_pype.tree_time_scale import iterative_timescale, plot_root_to_tip, plot_temporal_tree, tree_nodes_ci, temporal_pruning_sampler
from beast_pype.date_utilities import decimal_to_date
from Bio import Phylo, SeqIO
from copy import deepcopy
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import yaml
import os


In [ ]:
if metadata_path is None:
    if root_strain_names is None: 
        metadata_path = f'{save_dir}/metadata.csv'
    else:
        metadata_path = f'{save_dir}/metadata_with_root.csv'
    if not os.path.isfile(metadata_path):
        raise FileNotFoundError(f'If metadata_path is not given the the save_dir directory `{save_dir}` must contain either the file "metadata.csv" or the file "metadata_with_root.csv" (if root_strain_names is given). ')
        
if fasta_path is None:
    if root_strain_names is None: 
        fasta_path = f'{save_dir}/sequences.fasta'
    else:
        fasta_path = f'{save_dir}/sequences_with_root.fasta'
    if not os.path.isfile(fasta_path):
        raise FileNotFoundError(f'If fasta_path is not given the the save_dir directory `{save_dir}` must contain either the file "sequences.fasta" or the file "sequences_with_root.fasta" (if root_strain_names is given).')

## Iterative Clock Filtering and Time Tree Generation


In [ ]:
if root_strain_names is None:
    time_tree, all_outliers_df, fasta_path, metadata_path = iterative_timescale(
        ftree=f'{save_dir}/{tree_dir_name}/iqtree.nwk',
        falignment=fasta_path,
        fdates=metadata_path,
        remove_root=False,
        sample_id_field=sample_id_field,
        collection_date_field=collection_date_field,
        clock_filter=clock_filter,
        clock_filter_method=clock_filter_method,
        rng_seed=seed,
        max_iterations=max_iterations,
        remove_future_tips=remove_future_tips,
        clock_std=clock_std,
    )
else:
    time_tree, all_outliers_df, fasta_path, metadata_path = iterative_timescale(
        ftree=f'{save_dir}/{tree_dir_name}/iqtree.nwk',
        falignment=fasta_path,
        fdates=metadata_path,
        reroot=root_strain_names,
        remove_root=remove_root,
        sample_id_field=sample_id_field,
        collection_date_field=collection_date_field,
        clock_filter=clock_filter,
        clock_filter_method=clock_filter_method,
        rng_seed=seed,
        max_iterations=max_iterations,
        remove_future_tips=remove_future_tips,
        clock_std=clock_std,
    )

# Save all outliers across iterations
if not all_outliers_df.empty:
    all_outliers_df.to_csv(f'{save_dir}/{tree_dir_name}/clock_filter_removed_outliers.csv', index=False)
    # Save filtered fasta and metadata to standard locations
    import shutil
    final_fasta = f'{save_dir}/filtered_sequences.fasta'
    final_metadata = f'{save_dir}/filtered_metadata.csv'
    if fasta_path != final_fasta:
        shutil.copy2(fasta_path, final_fasta)
        fasta_path = final_fasta
    if metadata_path != final_metadata:
        shutil.copy2(metadata_path, final_metadata)
        metadata_path = final_metadata

# Save the final timetree
Phylo.write(time_tree.tree,
            f'{save_dir}/{tree_dir_name}/treetime.nwk',
            format='newick',
            format_branch_length='%1.8f')


Node confidence table

In [ ]:
node_ci_df = tree_nodes_ci(time_tree, fraction=0.95)
node_ci_df.to_csv(f'{save_dir}/{tree_dir_name}/treetime_node_confidence.csv', index=False)

Clock model stats

In [ ]:
clock_model = deepcopy(time_tree.clock_model)
rate = clock_model["slope"]
intercept = clock_model["intercept"]
r_val = clock_model.get("r_val", None)
r_sq = r_val**2 if r_val is not None else None
t_mrca = -intercept / rate if rate else None

clock_model["TMRCA root (year decimal)"] = t_mrca
clock_model["TMRCA root (date)"] = decimal_to_date(t_mrca).strftime("%Y-%m-%d") if t_mrca else None
clock_model["R^2 value"] = r_sq
clock_model["Correlation Coefficient (rho or r)"] = clock_model.pop("r_val", None)

yml_ready_clock_model = {}
for key, value in clock_model.items():
    if not isinstance(value, bool):
        try:
            value = float(value)
        except:
            value = str(value)
    yml_ready_clock_model[key] = value

with open(f"{save_dir}/{tree_dir_name}/treetime_clock_model_stats.yml", "w") as yaml_file:
    yaml.safe_dump(yml_ready_clock_model, yaml_file, indent=4)

# Build outliers_df with numdate and dist2root for root-to-tip plot
if not all_outliers_df.empty:
    from beast_pype.date_utilities import date_to_decimal
    if metadata_path.endswith(".tsv"):
        orig_meta = pd.read_csv(metadata_path, sep="	")
    else:
        orig_meta = pd.read_csv(metadata_path)
    id_col = sample_id_field if sample_id_field in orig_meta.columns else orig_meta.columns[0]
    date_col = collection_date_field if collection_date_field in orig_meta.columns else None
    if date_col:
        outlier_dates_df = orig_meta[[id_col, date_col]].rename(columns={id_col: "name", date_col: "date_str"})
        outliers_plot_df = all_outliers_df.merge(outlier_dates_df, on="name", how="left")
        outliers_plot_df["numdate"] = outliers_plot_df["date_str"].apply(
            lambda x: date_to_decimal(pd.to_datetime(x)) if pd.notna(x) else np.nan
        )
        outliers_plot_df["dist2root"] = rate * outliers_plot_df["numdate"] + intercept
    else:
        outliers_plot_df = all_outliers_df.copy()
        outliers_plot_df["numdate"] = np.nan
        outliers_plot_df["dist2root"] = np.nan
else:
    outliers_plot_df = None


## Root-to-Tip Plot

Including any outliers removed by clock filtering as orange dots.


In [ ]:
fig, ax = plot_root_to_tip(time_tree, outliers_df=outliers_plot_df, remove_future_tips=remove_future_tips)
fig.savefig(f"{save_dir}/{tree_dir_name}/treetime_root_to_tip.png", dpi=150, bbox_inches="tight")
plt.show()

## Plotting TreeTime tree

In [ ]:
# Time-scaled tree plot
fig_tree, ax_tree = plot_temporal_tree(time_tree)
fig_tree.savefig(f"{save_dir}/{tree_dir_name}/treetime_timetree.png", dpi=150, bbox_inches="tight")
plt.show()

## Downsampling time trees

### Obtaining strain ids and sequences.

Below, if the sample size is over suggested_downsample_tos then the normalised residuals from the root-to-tip regression above are used as weights in a probabilistic draw to remove leaves from a list of all the tips. The tips that are left to keep are stored in a list. See beast_pype.tree_time_scale.temporal_pruning_sampler for details on weighted removal of tips method.

In [ ]:
if downsample_to is not None:
    strain_sequences = SeqIO.parse(fasta_path, 'fasta')
    if metadata_path.endswith('.csv'):
        metadata_df = pd.read_csv(metadata_path, parse_dates=[collection_date_field])
    if metadata_path.endswith('.tsv'):
        metadata_df = pd.read_csv(metadata_path, sep='\t', parse_dates=[collection_date_field])
    tips = time_tree.tree.get_terminals()
    n_tips = len(tips)
    if downsample_to < n_tips:
        stuff_to_add = True
        str_downsample_to = str(downsample_to)
        sampled_ids = temporal_pruning_sampler(time_tree=time_tree, sample_size=downsample_to, seed=seed)
        selected_metadata = metadata_df[metadata_df[sample_id_field].isin(sampled_ids)]
        selected_seqs = [seq_record for seq_record in strain_sequences if seq_record.id in sampled_ids]
        selected_metadata.to_csv(f'{save_dir}/downsampled_metadata.csv', index=False)
        with open(f'{save_dir}/downsampled_sequences.fasta', 'w') as handle:
            SeqIO.write(selected_seqs, handle, 'fasta')
